In [2]:
import sys
import os

sibling_dir = os.path.abspath('../01_agentic_rag')

if sibling_dir not in sys.path:
    sys.path.append(sibling_dir)

In [3]:
from dotenv import load_dotenv
load_dotenv()

ModuleNotFoundError: No module named 'dotenv'

In [ ]:
# configure the OpenAI client with your API key
from openai import OpenAI

openai_client = OpenAI()

In [4]:
query1 = "I just discovered the course. Can I still join it?"
query2 = "I just found out about the program. Can I still enroll?"

In [12]:
# Choosing a model
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

ImportError: tokenizers>=0.22.0,<=0.23.0 is required for a normal functioning of this module, but found tokenizers==0.23.1.
Try: `pip install transformers -U` or `pip install -e '.[dev]'` if you're working with git main

In [ ]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [ ]:
q2 = "I just found out about the program. Can I still enroll?"
v2 = model.encode(q2)

In [ ]:
# encoding the document
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [ ]:
# Compare the query against the document using dot product
v1.dot(dv)

np.float32(0.32332397)

In [ ]:
v2.dot(dv)

np.float32(0.461237)

In [ ]:
q3 = "How to install Docker on Windows?"
v3 = model.encode(q3)

In [ ]:
v3.dot(dv)
# value is close to zero meaning that the query is not related to the document.


np.float32(0.019730438)

In [7]:
# loading data
from ingest import load_faq_data

documents = load_faq_data()

In [8]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [9]:
# generating embeddings for the documents

texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [10]:
texts[0]

"Course: When does the course start? A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."

In [11]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]


NameError: name 'model' is not defined

In [ ]:
vectors


NameError: name 'vectors' is not defined

In [ ]:
v1.dot(vectors[0])


np.float32(0.48740578)

In [ ]:
import numpy as np
X = np.array(vectors)
X

array([[-0.02670622, -0.12245756,  0.01594418, ..., -0.00230645,
        -0.11218391, -0.02365551],
       [-0.01099553, -0.11074745, -0.02536936, ...,  0.09022227,
        -0.02697366,  0.01975676],
       [-0.08896549, -0.06128184,  0.00775605, ...,  0.0405971 ,
         0.0047928 , -0.0274594 ],
       ...,
       [-0.03652925,  0.01415433, -0.06838643, ...,  0.04316785,
         0.08105534, -0.02148626],
       [-0.13091588, -0.06990597, -0.00931879, ..., -0.00044345,
        -0.01285729,  0.01426925],
       [-0.07984771,  0.01926989,  0.02544983, ..., -0.03368025,
        -0.01884021,  0.05837058]], shape=(1350, 384), dtype=float32)

In [ ]:
X.shape

(1350, 384)

In [ ]:
# similarity scores between the query vector v1 and all document vectors
scores = X.dot(v1)

In [ ]:
# np.argmax(scores) gives the index of the document with the highest similarity score 
# to the query vector v1.
idx=np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.762941))

In [ ]:
# document with the highest similarity score to the query vector v1
documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [ ]:
# taking the top 5 documents with the highest similarity scores to the query vector v1
top5 = np.argsort(-scores)[:5]
# top5 = top5[::-1]  # reverse the order to have the highest score first

In [ ]:
for i in top5:
    print(f"Score: {scores[i]:.4f}")
    print(f"section: {documents[i]['section']}")
    print(f"Question: {documents[i]['question']}")
    print(f"Answer: {documents[i]['answer']}")
    print()

Score: 0.7629
section: General Course-Related Questions
Question: Course: Can I still join the course after the start date?
Answer: Yes, even if you don't register, you're still eligible to submit the homework.

Be aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.

Score: 0.7579
section: General Course-Related Questions
Question: Course - Can I still join the course after the start date?
Answer: Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.

Be aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.

Score: 0.7192
section: General Course-Related Questions
Question: The course has already started. Can I still join it?
Answer: Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submi

In [ ]:
# creating a vector index for the documents using the minsearch library

from minsearch import VectorSearch

vec_index = VectorSearch(keyword_fields=["course"])
vec_index.fit(X, documents)

In [ ]:
# filtering the search results by course name using the vector index
vec_index.search(v1,num_results=5, filter_dict={"course": "llm-zoomcamp"})

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the for

In [ ]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [ ]:
from ingest import build_index

index = build_index(documents)

In [ ]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [ ]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes, but if you want to receive a certificate, you need to submit your project while they’re still accepting submissions.'

In [ ]:
# we subclass RAGBase and override search to encode the query before searching
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [ ]:
vector_assistant = RAGVector(
    embedder=model,
    index=vec_index,
    llm_client=openai_client,
)

In [ ]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes — you can still join after the program has begun. If you want a certificate, just make sure you submit your project while submissions are still open.'

In [ ]:
# creating a vector index for the documents using the sqlitesearch library

from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [ ]:
vs_index.fit(vectors, documents)

ValueError: Index already contains documents. Use clear() to reset the index or add() to append documents.

In [ ]:
# search for the most relevant documents using the vector index
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [ ]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section':

In [ ]:
# filtering the search results by course name using the vector index
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the co

In [ ]:
vs_index.close()